In [ ]:
import pandas as pd
import openpyxl

# Load the Excel file using openpyxl
file_path = '/Users/srivatsavkannan/Datasets/C-Spine Xray/datasets.xlsx'  # Replace with your actual file path
workbook = openpyxl.load_workbook(file_path)
sheet = workbook.active

# Define the column ranges and row indices
start_row = 3
end_row = 5002
disc_angle_columns = ['BH', 'BI', 'BJ', 'BK', 'BL']
cervical_slope_columns = ['CW', 'CX', 'CY', 'CZ', 'DA', 'DB']
other_columns = ['DC', 'DD', 'DE']
label_column = 'E'

# Combine all feature columns
feature_columns = disc_angle_columns + cervical_slope_columns + other_columns


# Helper function to convert column letter to index (1-based)
def col_to_index(col):
    return openpyxl.utils.cell.column_index_from_string(col)


# Initialize lists to hold the data
X = []
y = []

# Iterate through each row and extract the data
for row in range(start_row, end_row + 1):
    row_data = []

    # Extract feature data
    for col in feature_columns:
        cell_value = sheet.cell(row=row, column=col_to_index(col)).value
        if cell_value is None:
            cell_value = 0
        row_data.append(cell_value)

    # Append row data to X
   

    # Extract label data
    label_value = sheet.cell(row=row, column=col_to_index(label_column)).value
    if label_value is not None:
        X.append(row_data)
        label_value -= 1
        y.append(label_value)

# Convert lists to numpy arrays for model training
import numpy as np

X = np.array(X)
y = np.array(y)

# Print shapes of X and y to verify
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

print(y[0])




In [ ]:
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Verify the shapes of the split datasets
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit the scaler on the training data and transform both training and testing data
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Verify the scaling
print("First 5 rows of X_train after scaling:\n", X_train[:5])



In [ ]:
# Define the model as previously described
import tensorflow as tf
import matplotlib.pyplot as plt
NUM_FEATURES = X_train.shape[1]
NUM_CLASSES = len(set(y))  # Assuming y contains the class labels

# Input layer
inputs = tf.keras.layers.Input(shape=(NUM_FEATURES,))

# First Dense layer
x = tf.keras.layers.Dense(units=64, activation='relu')(inputs)
x = tf.keras.layers.Dropout(rate=0.5)(x)

# Second Dense layer
x = tf.keras.layers.Dense(units=128, activation='relu')(x)
x = tf.keras.layers.Dropout(rate=0.5)(x)

# Output layer
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

# Build the model
model = tf.keras.models.Model(inputs=inputs, outputs=outputs)

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Print the model summary
model.summary()

# Train the model
history = model.fit(X_train, y_train,  validation_data=(X_test, y_test), epochs=200, batch_size=32, validation_split=0.2)

model.save('curvature_quant.h5')
train_loss = history.history['loss']
val_loss = history.history['val_loss']
train_acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
epochs = range(1, len(train_loss) + 1)

# Plot training and validation loss
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs, train_loss, 'b', label='Training loss')
plt.plot(epochs, val_loss, 'r', label='Validation loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Plot training and validation accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, train_acc, 'b', label='Training accuracy')
plt.plot(epochs, val_acc, 'r', label='Validation accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

In [ ]:
# Evaluate the model on the test data
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, \
    confusion_matrix
import seaborn as sns
y_true = []
y_pred = []
idx = 0;
while idx < len(y_test):
    y_true.append(y_test[idx])
    y_pred.extend(np.argmax(model.predict(np.array([X_test[idx]])), axis=1))
    idx += 1
print(len(y_test))
# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate accuracy
accuracy = accuracy_score(y_true, y_pred)

# Calculate precision, recall, and F1-score
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

# Generate classification report
report = classification_report(y_true, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Classification Report:")
print(report)


cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
classes = ['Lordotic', 'Straight', 'Sigmoid1', 'Sigmoid2', 'Kyphotic']
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()